In [1]:
# ============================================================
# WEEK 1 — MDP Design & Custom Gym Environment
# Project 2: Travel & Hospitality — RL Dynamic Pricing
# Intern Branch: preeti-dev | Infotact Solutions
#
# UNIQUE FEATURES ADDED:
#   ★ External Market Events (holiday surge, competitor sale, weather)
#   ★ Three Customer Segments (business, leisure, last-minute)
#   ★ Competitor Pricing Signal in State Space
#   ★ Dual Seat Class (Economy + Business)
# ============================================================
 
 
# ── CELL 1: Install & Import Libraries ───────────────────
import subprocess, sys
 
def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
 
for lib in ["gymnasium", "numpy", "matplotlib", "seaborn", "pandas"]:
    install(lib)
 
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')
 
os.makedirs('../reports', exist_ok=True)
os.makedirs('../models',  exist_ok=True)
os.makedirs('../data',    exist_ok=True)
 
print("✅ All libraries imported successfully")
print(f"   Gymnasium version : {gym.__version__}")
 

✅ All libraries imported successfully
   Gymnasium version : 1.3.0


In [2]:
# ── CELL 2: MDP Problem Formulation ──────────────────────
print("=" * 60)
print("   ENHANCED MDP — Contextual Airline Dynamic Pricing")
print("=" * 60)
print()
print("  SCENARIO:")
print("  ─────────")
print("  An airline manages Economy + Business seats on a flight")
print("  departing in 30 days. Each day it sets prices for both")
print("  classes, facing 3 customer segments and random market")
print("  events that shift demand unexpectedly.")
print()
print("  STANDARD MDP COMPONENTS:")
print("  ─────────────────────────")
print("  State  : [eco_seats, biz_seats, days_left,")
print("            competitor_price_idx, market_event_id]")
print("  Action : [economy_price_idx, business_price_idx]")
print("  Reward : total daily revenue (eco + biz combined)")
print("  Done   : days_left = 0  OR  both classes sold out")
print()
print("  ★ UNIQUE ADDITIONS vs standard RL pricing projects:")
print("  ─────────────────────────────────────────────────────")
print("  ★ Market Events  : holiday/competitor sale/bad weather")
print("  ★ 3 Segments     : business / leisure / last-minute")
print("  ★ Competitor     : rival airline price in state space")
print("  ★ Dual Class     : Economy + Business managed together")
print()
print("✅ MDP formulation complete")


   ENHANCED MDP — Contextual Airline Dynamic Pricing

  SCENARIO:
  ─────────
  An airline manages Economy + Business seats on a flight
  departing in 30 days. Each day it sets prices for both
  classes, facing 3 customer segments and random market
  events that shift demand unexpectedly.

  STANDARD MDP COMPONENTS:
  ─────────────────────────
  State  : [eco_seats, biz_seats, days_left,
            competitor_price_idx, market_event_id]
  Action : [economy_price_idx, business_price_idx]
  Reward : total daily revenue (eco + biz combined)
  Done   : days_left = 0  OR  both classes sold out

  ★ UNIQUE ADDITIONS vs standard RL pricing projects:
  ─────────────────────────────────────────────────────
  ★ Market Events  : holiday/competitor sale/bad weather
  ★ 3 Segments     : business / leisure / last-minute
  ★ Competitor     : rival airline price in state space
  ★ Dual Class     : Economy + Business managed together

✅ MDP formulation complete


In [3]:
# ── CELL 3: Market Event System ──────────────────────────
# Each day, one of 4 possible market conditions applies.
# This is the first unique feature — real airline pricing
# reacts to external shocks, not just internal inventory.
 
MARKET_EVENTS = {
    0: {'name': 'Normal',          'demand_multiplier': 1.0,  'color': '#60a5fa'},
    1: {'name': 'Holiday Surge',   'demand_multiplier': 1.6,  'color': '#34d399'},
    2: {'name': 'Competitor Sale', 'demand_multiplier': 0.6,  'color': '#f87171'},
    3: {'name': 'Bad Weather',     'demand_multiplier': 0.75, 'color': '#fbbf24'},
}
 
# Probability of each event occurring on any given day
EVENT_PROBS = [0.65, 0.12, 0.15, 0.08]  # must sum to 1.0
 
print("✅ Market Event System defined")
print()
print(f"  {'Event':<22} {'Demand Multiplier':>18} {'Probability':>12}")
print("  " + "-" * 54)
for idx, ev in MARKET_EVENTS.items():
    print(f"  {ev['name']:<22} {ev['demand_multiplier']:>18.1f}×"
          f" {EVENT_PROBS[idx]:>11.0%}")


✅ Market Event System defined

  Event                   Demand Multiplier  Probability
  ------------------------------------------------------
  Normal                                1.0×         65%
  Holiday Surge                         1.6×         12%
  Competitor Sale                       0.6×         15%
  Bad Weather                           0.8×          8%


In [ ]:
# ── CELL 4: Customer Segment System ──────────────────────
# Three distinct customer types exist simultaneously.
# Each has different price sensitivity and booking timing.
# This is the second unique feature.
 
CUSTOMER_SEGMENTS = {
    'business': {
        'price_sensitivity' : 0.3,   # low — will pay high prices
        'booking_window'    : 'any', # books anytime
        'base_probability'  : 0.25,  # 25% of daily traffic
        'color'             : '#a78bfa'
    },
    'leisure': {
        'price_sensitivity' : 0.9,   # high — very price conscious
        'booking_window'    : 'early',# books far in advance
        'base_probability'  : 0.50,  # 50% of daily traffic
        'color'             : '#60a5fa'
    },
    'last_minute': {
        'price_sensitivity' : 0.2,   # very low — desperate, pays anything
        'booking_window'    : 'late', # only books in last 7 days
        'base_probability'  : 0.25,  # 25% of daily traffic
        'color'             : '#f87171'
    }
}
 
print("✅ Customer Segment System defined")
print()
print(f"  {'Segment':<15} {'Price Sensitivity':>18} "
      f"{'Base Prob':>10} {'When They Book':>15}")
print("  " + "-" * 60)
for name, seg in CUSTOMER_SEGMENTS.items():
    print(f"  {name:<15} {seg['price_sensitivity']:>18.1f} "
          f"{seg['base_probability']:>10.0%} {seg['booking_window']:>15}")
